In [1]:
import numpy as np
import pandas as pd

# Plantilla

Es necesario ajustar las definiciones, las fuentes de los datos y posiblemente definiciones si la ENEMDU tiene una dimensión geográfica y temporal al mismo tiempo

In [12]:
# data = pd.read_stata(r"datos/ECU_2004m12_BID.dta", convert_categoricals=False) # para bases de stata

data9 = pd.read_spss(r"C:\Users\oscarj\OneDrive - Inter-American Development Bank Group\Desktop\ecu\2020\BDD_ENEMDU_2020_09_SPSS\enemdu_personas_2020_09.sav", convert_categoricals=False) # para bases de stata
data12 = pd.read_spss(r"C:\Users\oscarj\OneDrive - Inter-American Development Bank Group\Desktop\ecu\2020\BBDD_PUBLICACION_ DIC 20_SPSS\BBDD_PUBLICACION_ DIC 20_SPSS\enemdu_persona_2020_12.sav", convert_categoricals=False) # para bases de stata

In [13]:
data9.columns = [x.lower() for x in data9.columns]
data12.columns = [x.lower() for x in data12.columns]

## Revisar los datos

| septiembre | diciembre |
|-----------|-----------|
| area  | area  |
|  | ciudad  |
|  | panelm  |
| vivienda  | vivienda  |
| hogar  | hogar  |
| p02  | p02  |
| p03  | p03  |
| p66  | p66  |
| fexp  | fexp  |
| p20  | p20  |
| id_hogar  | id_hogar  |

En esta encuesta tenemos separadas cuatro diferentes bases para cada trimestre, esto cambia la lógica que habíamos tenido hasta ahora así que de aquí en adelante cambiamos algo del código, mantenemos de acuerdo a las etiquetas de las variables pe63 como la variable de ingreso laboral monetario de la actividad principal asalariada para mantener la concordancia con el resto de los años.

En ests encuesta de 2018, los trimestres 3 y cuatro no tienen la variable de ciudad, zona y sector en este caso asignaremos el IPC nacional y agregamos una variable de id de hogar propia de la encuesta

Hay variables dicotomicas para cada mes (ene, feb, mar, abr, may, jun, jul, ago, sep, oct, nov, dic) que dicen si estuvo o no trabajando, estas variables las usamos antes para identificar la condición de trabajo para diferentes meses en encuestas anuales o incompletas donde asumíamos que mantenía el mismo salario si estaba ocupado en ese mes, sin mbargo estas variables tenían el problema de no corresponder de forma exacta con el año o mes de la encuesta. Ahora sin embargo podemos cambiar las suposiciones y solamente asumir que si la variable 'trabajando' que pregunta si el individuo trabajó la semana pasada se cumple vamos a asumir que trabajo durante todo el trimestre, de esta manera podemos mejorar las suposiciones de ocupación mensual, mantenemos la idea de que si el individuo trabajo recibe su ingreso laboral reportado.

In [19]:
columnas = pd.Index(['area', 'ciudad', 'panelm',
            'vivienda', 'hogar', 'p66',
            'fexp', 'p02', 'p03', 'p20', 'id_hogar'])

Filtramos solo las columnas de interés para alivar el peso en la memoria

In [20]:
data9 = data9[columnas.intersection(data9.columns)]
data12 = data12[columnas.intersection(data12.columns)]

Creamos una variable de ingreso laboral que es igual al ingreso por asalariado y limpiamos según los valores de ingrl, para mantener ambas variables para cada base consistente

In [23]:
data12['p66'].value_counts().sort_index(ascending=False)

p66
999999.0     79
5000.0        2
4000.0        3
3500.0        1
3440.0        1
           ... 
20.0          5
17.0          1
12.0          2
10.0          4
0.0         166
Name: count, Length: 401, dtype: int64

In [24]:
data9['p66'] = pd.to_numeric(data9['p66'], errors='coerce')
data9['p66'] = data9['p66'].apply(lambda x: np.nan if x > 5000 else x)
data9['p66'] = data9['p66'].apply(lambda x: np.nan if x < 0 else x)

data12['p66'] = pd.to_numeric(data12['p66'], errors='coerce')
data12['p66'] = data12['p66'].apply(lambda x: np.nan if x > 5000 else x)
data12['p66'] = data12['p66'].apply(lambda x: np.nan if x < 0 else x)

C:\Users\oscarj\AppData\Local\Temp\ipykernel_17748\1323725933.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data12['p66'] = pd.to_numeric(data12['p66'], errors='coerce')
C:\Users\oscarj\AppData\Local\Temp\ipykernel_17748\1323725933.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data12['p66'] = data12['p66'].apply(lambda x: np.nan if x > 5000 else x)
C:\Users\oscarj\AppData\Local\Temp\ipykernel_17748\1323725933.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a 

In [25]:
data9['ingr'] = data9['p66']
data12['ingr'] = data12['p66']

C:\Users\oscarj\AppData\Local\Temp\ipykernel_17748\1257734763.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data12['ingr'] = data12['p66']


Ingreso mensual asumiendo que las personas reciben el mismo valor reportado en 'ingr' siempre que reportan estar ocupados la semana pasada, de acuerdo a la variable 'trabajo'

In [26]:
data9['ingr_t3'] = data9.apply(lambda x: x['ingr'] if x['p20'] == 1 else np.nan, axis=1)

data12['ingr_t4'] = data12.apply(lambda x: x['ingr'] if x['p20'] == 1 else np.nan, axis=1)

C:\Users\oscarj\AppData\Local\Temp\ipykernel_17748\508849827.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data12['ingr_t4'] = data12.apply(lambda x: x['ingr'] if x['p20'] == 1 else np.nan, axis=1)


## Deflactamos y transformamos el ingreso

Esto deja todo en dólares constantes de 2014

In [27]:
# Carga base de datos con ipc
data_externa = pd.read_excel("data_externa.xlsx", sheet_name='datos')

# filtra año de interés
datos_actual = data_externa[data_externa['Año'] == 2020]
datos_base = data_externa[data_externa['Año'] == 2014]

Diccionarios de ipc

In [28]:
ipc_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Sierra': fila['Sierra'],
        'Costa': fila['Costa'],
        'Guayaquil': fila['Guayaquil'],
        'Esmeraldas': fila['Esmeraldas'],
        'Machala': fila['Machala'],
        'Manta': fila['Manta'],
        'Quito': fila['Quito'],
        'Loja': fila['Loja'],
        'Cuenca': fila['Cuenca'],
        'Ambato': fila['Ambato']
    }
     for _, fila in datos_actual.iterrows()
     }

ipc_base_dict = {fila['trimestre']: {
        'Nacional': fila['Nacional'],
        'Sierra': fila['Sierra'],
        'Costa': fila['Costa'],
        'Guayaquil': fila['Guayaquil'],
        'Esmeraldas': fila['Esmeraldas'],
        'Machala': fila['Machala'],
        'Manta': fila['Manta'],
        'Quito': fila['Quito'],
        'Loja': fila['Loja'],
        'Cuenca': fila['Cuenca'],
        'Ambato': fila['Ambato']
    }
     for _, fila in datos_base.iterrows()
     }

### Creamos identificadores para las ciudades siguiendo los códigos del INEC y para los trimestres

In [30]:
# Corregimos los códigos para usarlos cómo texto
data12['ciudad'] = data12['ciudad'].apply(str)
data12['ciudad'] = data12['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)
data12['ciudad_2'] = data12['ciudad'].apply(lambda x: x[:4])

C:\Users\oscarj\AppData\Local\Temp\ipykernel_17748\1582966245.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data12['ciudad'] = data12['ciudad'].apply(str)
C:\Users\oscarj\AppData\Local\Temp\ipykernel_17748\1582966245.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data12['ciudad'] = data12['ciudad'].apply(lambda x: '0' + x if len(x) == 5 else x)
C:\Users\oscarj\AppData\Local\Temp\ipykernel_17748\1582966245.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataF

Diccionario ciudades disponibles

In [31]:
parroquia_dict = {
    '0101': 'Cuenca',
    '0901': 'Guayaquil',
    '0801': 'Esmeraldas',
    '0701': 'Machala',
    '1308': 'Manta',
    '1701': 'Quito',
    '1101': 'Loja',
    '1801': 'Ambato'
}

def get_parroquia(codigo):
    if codigo in parroquia_dict:
        return parroquia_dict[codigo]
    elif codigo[:2] in ['01', '02', '03', '04', '05', '06', '10', '11', '17', '18']:
        return 'Sierra'
    elif codigo[:2] in ['07', '08', '09', '12', '13', '23', '24']:
        return 'Costa'
    else:
        return 'Nacional'

data9['ciudad_asignada'] = 'Nacional'
data12['ciudad_asignada'] = data12['ciudad_2'].apply(get_parroquia)

C:\Users\oscarj\AppData\Local\Temp\ipykernel_17748\56610171.py:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data12['ciudad_asignada'] = data12['ciudad_2'].apply(get_parroquia)


In [34]:
data9['ciudad_asignada'].value_counts()

ciudad_asignada
Nacional    30317
Name: count, dtype: int64

### Asignamos el ipc correspondiente según ciudad correspondiente

$\begin{equation}
    ingr_{USD-base-2014}^{i} = ingr_{dólares}^{i}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Desde 2000 en adelante ya no es necesario utilizar el tipo de cambio debido al cambio de moneda

In [35]:
# Función que asigna valores correspondientes
def asigna_ipc(fila, trimestre):
    return ipc_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

def asigna_ipc_base(fila, trimestre):
    return ipc_base_dict.get(trimestre, {}).get(fila['ciudad_asignada'], None)

In [36]:
data9['ipc_t3'] = data9.apply(lambda fila: asigna_ipc(fila, 3), axis=1)
data9['ipc_base_t3'] = data9.apply(lambda fila: asigna_ipc_base(fila, 3), axis=1)

data12['ipc_t4'] = data12.apply(lambda fila: asigna_ipc(fila, 4), axis=1)
data12['ipc_base_t4'] = data12.apply(lambda fila: asigna_ipc_base(fila, 4), axis=1)

C:\Users\oscarj\AppData\Local\Temp\ipykernel_17748\3596560344.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data12['ipc_t4'] = data12.apply(lambda fila: asigna_ipc(fila, 4), axis=1)
C:\Users\oscarj\AppData\Local\Temp\ipykernel_17748\3596560344.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data12['ipc_base_t4'] = data12.apply(lambda fila: asigna_ipc_base(fila, 4), axis=1)


In [37]:
# Calculamos el deflactor
data9['def_t3'] = (data9['ipc_base_t3'] / data9['ipc_t3'])
data12['def_t4'] = (data12['ipc_base_t4'] / data12['ipc_t4'])

C:\Users\oscarj\AppData\Local\Temp\ipykernel_17748\713181605.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data12['def_t4'] = (data12['ipc_base_t4'] / data12['ipc_t4'])


Ingreso promedio en el trimeste

In [38]:
data9['ingr_t3_r'] = data9['ingr_t3'] * data9['def_t3']
data12['ingr_t4_r'] = data12['ingr_t4'] * data12['def_t4']

C:\Users\oscarj\AppData\Local\Temp\ipykernel_17748\2295724162.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data12['ingr_t4_r'] = data12['ingr_t4'] * data12['def_t4']


In [39]:
print(data9['ingr_t3_r'].mean())
print(data12['ingr_t4_r'].mean())

447.477380144484
429.02905588872636


## Calculo ingreso de los hogares

In [40]:
data9['idef_hogar'] = data9['id_hogar']
data12['idef_hogar'] = data12['id_hogar']

print(len(data9['idef_hogar'].unique()))
print(len(data12['idef_hogar'].unique()))

8587
8756


C:\Users\oscarj\AppData\Local\Temp\ipykernel_17748\862188344.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data12['idef_hogar'] = data12['id_hogar']


Si todos los miembros del hogar tienen NA como ingreso, mantener NA, si al menos uno tiene un ingreso sumamos para el ingreso del hogar, así evitamos subestimar el ingreso del hogar si tenemos valores perdidos

In [41]:
# Definimos una función que sume pero devuelva NA si todos son NA
def sum_with_na(series):
    if series.isna().all():
        return pd.NA
    else:
        return series.sum(skipna=True)

In [42]:
data9['ingr_t3_h'] = data9.groupby('idef_hogar')['ingr_t3_r'].transform(sum_with_na)
data12['ingr_t4_h'] = data12.groupby('idef_hogar')['ingr_t4_r'].transform(sum_with_na)

C:\Users\oscarj\AppData\Local\Temp\ipykernel_17748\2932482167.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data12['ingr_t4_h'] = data12.groupby('idef_hogar')['ingr_t4_r'].transform(sum_with_na)


In [43]:
print(data9['ingr_t3_h'].mean())
print(data12['ingr_t4_h'].mean())

641.7326321216287
620.3402196252964


## Sacamos edades negativas y mayores a 100 años

In [44]:
print(len(data9))
print(len(data12))

30317
30646


Transformamos las variables de edad a numericas para evitar problemas

In [45]:
data9['edad'] = pd.to_numeric(data9['p03'], errors='coerce')
data12['edad'] = pd.to_numeric(data12['p03'], errors='coerce')

C:\Users\oscarj\AppData\Local\Temp\ipykernel_17748\937412886.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data12['edad'] = pd.to_numeric(data12['p03'], errors='coerce')


In [46]:
data9 = data9.loc[(data9['edad'] >= 0) & (data9['edad'] < 100)]
data12 = data12.loc[(data12['edad'] >= 0) & (data12['edad'] < 100)]

## Ingreso individual descontando cargas familiares

Utilizando la metodología del autor dividimos el ingreso del hogar para la escala $(A_{i}+kC_{i})^{s}$ donde $A_{i}$ es al número de adultos, $C_{i}$ es el número de niños en el hogar $i$. $k$ es el costo en recursos de cada niño y $s$ busca reflejar las restricciones

In [47]:
k = 0.4
s = 0.9

In [48]:
# Si es necesario calcular el número de niños
data9['es_nino'] = data9['edad'] < 10
data9['ninos'] = data9.groupby('idef_hogar')['es_nino'].transform('sum')

data12['es_nino'] = data12['edad'] < 10
data12['ninos'] = data12.groupby('idef_hogar')['es_nino'].transform('sum')

# Si es necesario calcular el número de adultos
data9['es_adulto'] = data9['edad'] > 10
data9['adultos'] = data9.groupby('idef_hogar')['es_adulto'].transform('sum')

data12['es_adulto'] = data12['edad'] > 10
data12['adultos'] = data12.groupby('idef_hogar')['es_adulto'].transform('sum')

In [49]:
data9['escala'] = (data9['adultos'] + k * data9['ninos']) ** s
data12['escala'] = (data12['adultos'] + k * data12['ninos']) ** s

In [50]:
data9['ingr_t_t3'] = data9['ingr_t3_h'] / data9['escala']
data12['ingr_t_t4'] = data12['ingr_t4_h'] / data12['escala']

In [51]:
print(data9['ingr_t_t3'].mean())
print(data12['ingr_t_t4'].mean())

192.95778210428568
190.5424639625443


## Umbrales de pobreza

Incluimos los índices de pobreza si es posible a nivel regional para luego poder utilizar de mejor forma el factor de expansión

$\begin{equation}
    umbral_{USD-base-2014}^{i} = umbral_{año}\left( \frac{ipc^{i}_{2014}}{ipc^{i}_{año}}\right)
\end{equation}$

Con $i$ el trimestre de interés y $año$ el año de interés

Diccionario de umbral

In [52]:
umbral_dict = dict(zip(datos_actual['trimestre'], datos_actual['umbral de pobreza']))

## Cálculo del índice de pobreza de Foster, Greer y Thorbecke

Para calcular un ínidce de pobreza se utiliza a Foster, Greer y Thorbecke (1984), ya que satisface algunas caracterísitcas de distribución que son positivas e igual a las enunciadas por Sen, el autor usa el mismo índice.

$\begin{equation}FGT_{\alpha} = \frac{1}{N}\sum_{i=1}^{H}\left(\frac{z-y_{i}}{z}\right)^{\alpha}\end{equation}$

Donde $z$ es el umbral de pobreza, $N$ es el número de personas en la economía, $H$ es el número de pobres (personas debajo de la línea de pobreza) $y_{i}$ es el ingreso de cada individuo. Mientras mayor es el valor de $\alpha$ mayor es el peso de los individuos más pobres, mayor $FGT$ mayor pobreza en la economía.

En este caso los umbrales están anivel nacional, aún así buscamos calcular la pobreza por región y sacar un promedio ponderado por región para la pobreza nacional, con el objetivo de hacerlo más específico

In [53]:
datos_final = pd.DataFrame(index=['t1', 't2', 't3', 't4'], columns=['fgt0', 'fgt1', 'fgt2', 'a25', 'a50', 'a75', 'ingreso_promedio'])

In [54]:
data9['persona_fexp'] = 1 * data9['fexp']
data12['persona_fexp'] = 1 * data12['fexp']

In [55]:
dict_t = {3:data9, 4:data12}

In [56]:
for t in [3, 4]:
    col_ingr = f'ingr_t_t{t}'
    col_pobres = f'pobres_t{t}'

    # una columna que identifica a quienes están por debajo de la línea de pobreza por trimestre
    dict_t[t][col_pobres] = (
        (dict_t[t][col_ingr] - umbral_dict.get(t)) < 0
    ).astype(int)

In [57]:
print("pobreza t3: ", (data9['pobres_t3'] * data9['fexp']).sum()/data9.loc[data9['ingr_t_t3'] >= 0]['persona_fexp'].sum())
print("pobreza t4: ", (data12['pobres_t4'] * data12['fexp']).sum()/data12.loc[data12['ingr_t_t4'] >= 0]['persona_fexp'].sum())

pobreza t3:  0.25174646843147225
pobreza t4:  0.269103282137662


In [58]:
for t in [3, 4]:
    # Filtramos para cada trimestre
    df_temp = dict_t[t].loc[dict_t[t][f'ingr_t_t{t}'] >= 0].copy()

    # Calculamos una columna de pobres
    df_temp['pobres'] = (df_temp[f'ingr_t_t{t}'] - umbral_dict[t]) < 0

    # Ratio de pobres sobre el total
    ratio = (umbral_dict[t] - df_temp[f'ingr_t_t{t}']) / umbral_dict[t]

    # Calculamos el índice para alpha 0, 1 y 2 solo donde 'pobres' == True.
    for i in range(3):
        col = f'fgt{i}'
        df_temp[col] = np.where(df_temp['pobres'], ratio**i, 0)

    # Cálculo del índice ponderado: se usa el factor de expansión como peso
    peso_total = df_temp['fexp'].sum()
    fgt0 = (df_temp['fgt0'] * df_temp['fexp']).sum() / peso_total
    fgt1 = (df_temp['fgt1'] * df_temp['fexp']).sum() / peso_total
    fgt2 = (df_temp['fgt2'] * df_temp['fexp']).sum() / peso_total
    
    # Guardamos los resultados
    datos_final.loc[f't{t}', 'fgt0'] = fgt0
    datos_final.loc[f't{t}', 'fgt1'] = fgt1
    datos_final.loc[f't{t}', 'fgt2'] = fgt2

In [60]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
t2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
t3,0.251746,0.113112,0.075535,NaN,NaN,NaN,NaN
t4,0.269103,0.105264,0.060092,NaN,NaN,NaN,NaN


## Calculo del índice de desigualdad de Atkinson

Vamos a calcular el índice de desigualdad de atkinson con un parámetro $\epsilon$ de aversión a la desigualdad y un $\mu$ que es igual a la media de los ingresos individuales, con la siguiente fórmula.

$\begin{equation}A = 1-\frac{1}{\mu}\left(\frac{1}{N}\sum_{i=1}^{N}y^{1-\epsilon}\right)^{1/(1-\epsilon)}\end{equation}$

Donde $y_{i}$ es el ingreso individual y $\mu$ es el ingreso medio

In [61]:
for t in [3, 4]:
    # Filtramos para cada trimestre
    df_temp = dict_t[t].loc[dict_t[t][f'ingr_t_t{t}'] >= 0].copy()

    # Suma total de los factores de expansión para el trimestre
    peso_total = df_temp['fexp'].sum()
    
    # Ingreso promedio ponderado
    mu = (df_temp[f'ingr_t_t{t}'] * df_temp['fexp']).sum() / peso_total

    # Calculamos el índice A para epsilon 0.25, 0.5 y 0.75 utilizando los pesos
    indices = {}
    for i in [0.25, 0.5, 0.75]:
        A_i = ((df_temp[f'ingr_t_t{t}']**(1-i) * df_temp['fexp']).sum() / peso_total)**(1/(1-i))
        indices[i] = A_i

    # Ratio de pobreza con el índice total (aplicando la fórmula)
    a25 = 1 - 1/mu * indices[0.25]
    a50 = 1 - 1/mu * indices[0.5]
    a75 = 1 - 1/mu * indices[0.75]

    # Guardamos los resultados
    datos_final.loc[f't{t}', 'a25'] = a25
    datos_final.loc[f't{t}', 'a50'] = a50
    datos_final.loc[f't{t}', 'a75'] = a75


In [63]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
t2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
t3,0.251746,0.113112,0.075535,0.095742,0.18973,0.303658,NaN
t4,0.269103,0.105264,0.060092,0.08525,0.1641,0.246085,NaN


Guardamos el ingreso promedio

In [64]:
for t in [3, 4]:
    df_temp = dict_t[t].loc[dict_t[t][f'ingr_t_t{t}'] >= 0].copy()
    
    # Calcula la suma total de los factores de expansión
    peso_total = df_temp['fexp'].sum()
    
    # Calcula el ingreso promedio ponderado
    media_ponderada = (df_temp[f'ingr_t_t{t}'] * df_temp['fexp']).sum() / peso_total
    
    datos_final.loc[f't{t}', 'ingreso_promedio'] = media_ponderada

In [65]:
datos_final

,fgt0,fgt1,fgt2,a25,a50,a75,ingreso_promedio
t1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
t2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
t3,0.251746,0.113112,0.075535,0.095742,0.18973,0.303658,188.690869
t4,0.269103,0.105264,0.060092,0.08525,0.1641,0.246085,173.806208


### Inserta los cálculos en la base final

In [66]:
indices = pd.read_csv("indices.csv", encoding='latin-1')

In [67]:
ano = 2020
# Asegurar que el índice de datos_final coincide con trimestres 1..4
datos_final = datos_final.copy()
datos_final["trimestre"] = [1, 2, 3, 4]
datos_final["Año"] = ano

# Reemplazar en indices usando mask
for col in ["fgt0","fgt1","fgt2","a25","a50","a75","ingreso_promedio"]:
    indices.loc[indices["Año"].eq(ano), col] = datos_final[col].values

C:\Users\oscarj\AppData\Local\Temp\ipykernel_17748\781764880.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan 0.2517464684314711 0.26910328213765955]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  indices.loc[indices["Año"].eq(ano), col] = datos_final[col].values
C:\Users\oscarj\AppData\Local\Temp\ipykernel_17748\781764880.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[nan nan 0.11311202214094634 0.10526443280340791]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  indices.loc[indices["Año"].eq(ano), col] = datos_final[col].values
C:\Users\oscarj\AppData\Local\Temp\ipykernel_17748\781764880.py:9: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Va

In [68]:
indices.to_csv('indices.csv', encoding='latin-1', index=None)